# 02 – Tail Labelling

Inspect the binary tail-event labels generated by `preprocess.py`.
Examines the tail-event rate, clustering behaviour, and per-asset
contribution to tail days.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from omegaconf import OmegaConf

cfg = OmegaConf.load('../configs/data.yaml')
processed_dir = Path('../') / cfg.processed_dir

## 1. Load labels and returns

In [ ]:
labels = pd.read_csv(processed_dir / 'tail_labels.csv', index_col=0, parse_dates=True).squeeze()
returns = pd.read_csv(processed_dir / 'returns_raw.csv', index_col=0, parse_dates=True)

print(f'Total days:     {len(labels)}')
print(f'Tail events:    {labels.sum()} ({100*labels.mean():.1f}%)')
print(f'Normal days:    {(1-labels).sum()}')

## 2. Tail events over time

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# SPY return
ax1.plot(returns.index, returns.iloc[:, 0], linewidth=0.6, color='steelblue')
tail_idx = labels[labels == 1].index
ax1.scatter(tail_idx, returns.loc[tail_idx].iloc[:, 0],
            color='red', s=8, zorder=5, label='Tail event')
ax1.set_ylabel(f'{returns.columns[0]} log-return')
ax1.legend()

# Rolling tail rate (60-day)
rolling_tail_rate = labels.rolling(60).mean()
ax2.plot(rolling_tail_rate.index, rolling_tail_rate, color='tomato', linewidth=0.8)
ax2.axhline(cfg.tail_quantile, color='grey', linestyle='--', linewidth=0.8, label=f'Target {cfg.tail_quantile:.0%}')
ax2.set_ylabel('60-day tail rate')
ax2.set_xlabel('Date')
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle('Tail events over time', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/tail_events_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Per-asset tail contribution

In [ ]:
tail_days = returns.loc[labels == 1]
threshold = returns.quantile(cfg.tail_quantile)
contrib = (tail_days <= threshold).sum() / len(tail_days)

fig, ax = plt.subplots(figsize=(7, 4))
contrib.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Fraction of tail days where asset is below quantile')
ax.set_title('Per-asset tail contribution')
plt.tight_layout()
plt.savefig('../outputs/figures/per_asset_tail_contribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(contrib.to_string())

## 4. Return distribution: tail vs. normal days

In [ ]:
normal_days = returns.loc[labels == 0]

fig, axes = plt.subplots(1, len(returns.columns), figsize=(4 * len(returns.columns), 4))
for ax, col in zip(axes, returns.columns):
    ax.hist(normal_days[col].dropna(), bins=60, density=True, alpha=0.5,
            color='steelblue', label='Normal')
    ax.hist(tail_days[col].dropna(), bins=30, density=True, alpha=0.6,
            color='red', label='Tail')
    ax.set_title(col)
    ax.set_xlabel('Log-return')
axes[0].legend()

fig.suptitle('Return distribution: tail vs. normal', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/tail_vs_normal_dist.png', dpi=150, bbox_inches='tight')
plt.show()